# Time is the Hardest Dimension

Time dimensions are special: they're naturally ordered, and they need an aggregation granularity (daily, weekly, monthly). This makes them the foundation for a family of transforms that are simple to express but non-trivial to implement in SQL.

This notebook demonstrates SLayer's time-related features with working code.

See also: [Expressions](../../concepts/expressions.md) | [Queries](../../concepts/queries.md)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, datasets = ensure_jaffle_shop()

## Time Dimensions and Granularity

A time dimension truncates a date/timestamp field to a given granularity. SLayer supports `day`, `week`, `month`, `quarter`, and `year`.

Let's start with monthly revenue:

In [2]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "ordered_at", "granularity": "month"}],
        "metrics": ["sum(order_total)"],
        "order": [{"field": "ordered_at", "direction": "asc"}],
        "limit": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>14}")
print("-" * 28)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    print(f"{month:<12} ${rev:>13,.2f}")

Month               Revenue
----------------------------
2023-11      $     5,161.96
2023-12      $    22,129.59
2024-01      $    26,290.80
2024-02      $    33,117.83
2024-03      $    43,950.73
2024-04      $    47,220.19
2024-05      $    53,072.01
2024-06      $    91,784.53
2024-07      $    95,886.29
2024-08      $   107,840.98
2024-09      $   100,250.49
2024-10      $   106,235.18


## Month-over-Month Change

The `change()` transform computes the difference from the previous period. It needs no extra arguments — the time dimension and granularity are already known from the query. Internally it uses a self-join CTE (not LAG), so it handles time gaps correctly: the previous period is matched on the calendar (e.g., the month before), not on the previous *row* in the result set. The very first period in the window still has no prior period to compare against, so its change is NULL (the first row below shows `-`); subsequent periods are gap-safe.

`change_pct()` gives the percentage change:

In [3]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "ordered_at", "granularity": "month"}],
        "metrics": [
            "sum(order_total)",
            {"metric": "change(sum(order_total))", "alias": "mom_change"},
            {"metric": "change_pct(sum(order_total))", "alias": "mom_pct"},
        ],
        "order": [{"field": "ordered_at", "direction": "asc"}],
        "limit": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'MoM Change':>14} {'MoM %':>8}")
print("-" * 52)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    pct = row["orders.mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>13,.2f} {chg_str:>14} {pct_str:>8}")

Month               Revenue     MoM Change    MoM %
----------------------------------------------------
2023-11      $     5,161.96              -        -
2023-12      $    22,129.59       $+16,968  +328.7%
2024-01      $    26,290.80        $+4,161   +18.8%
2024-02      $    33,117.83        $+6,827   +26.0%
2024-03      $    43,950.73       $+10,833   +32.7%
2024-04      $    47,220.19        $+3,269    +7.4%
2024-05      $    53,072.01        $+5,852   +12.4%
2024-06      $    91,784.53       $+38,713   +72.9%
2024-07      $    95,886.29        $+4,102    +4.5%
2024-08      $   107,840.98       $+11,955   +12.5%
2024-09      $   100,250.49        $-7,590    -7.0%
2024-10      $   106,235.18        $+5,985    +6.0%


## Year-over-Year with time_shift

`time_shift(metric, offset, granularity)` shifts the time dimension by the given offset before applying the query's granularity. This is implemented as a calendar-based self-join CTE — semantically simple, but the SQL is non-trivial.

For example, `time_shift(order_total_sum, -1, 'year')` gives "same month, previous year":

In [4]:
import datetime as dt

# Use last full calendar year so the example stays in the demo's data window
# (the loader shifts dates so MAX(ordered_at) == today, with ~4 years of history).
last_year = dt.date.today().year - 1
date_range = [f"{last_year}-01-01", f"{last_year}-12-31"]

result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [
            {
                "field": "ordered_at",
                "granularity": "month",
                "date_range": date_range,
            }
        ],
        "metrics": [
            "sum(order_total)",
            {"metric": "time_shift(sum(order_total), -1, 'year')", "alias": "prev_year"},
        ],
        "order": [{"field": "ordered_at", "direction": "asc"}],
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'Prev Year':>14}")
print("-" * 42)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    prev = row["orders.prev_year"]
    prev_str = f"${prev:>13,.2f}" if prev is not None else f"{'n/a':>14}"
    print(f"{month:<12} ${rev:>13,.2f} {prev_str}")

Month               Revenue      Prev Year
------------------------------------------
2025-01      $   168,371.90 $    26,290.80
2025-02      $   162,558.46 $    33,117.83
2025-03      $   185,861.95 $    43,950.73
2025-04      $   182,862.22 $    47,220.19
2025-05      $   195,714.15 $    53,072.01
2025-06      $   180,934.33 $    91,784.53
2025-07      $   203,437.62 $    95,886.29
2025-08      $   246,156.86 $   107,840.98
2025-09      $   203,297.96 $   100,250.49
2025-10      $   212,511.05 $   106,235.18
2025-11      $   223,349.31 $   112,402.81
2025-12      $   302,226.77 $   148,952.13


### time_shift vs lag

`time_shift` always uses a calendar-based self-join, with or without a granularity argument. Without one, `time_shift(x, -1)` shifts by one bucket of the query's own time granularity; with one (like `'year'` above), the shift can be any calendar amount — not just a multiple of the query granularity. Either way it's gap-safe: the previous value is matched on the actual calendar date, so a missing intermediate month produces NULL rather than the wrong value. The join also matches on all non-time dimensions, so grouped series (e.g. per-store) reset cleanly per group.

`lag(x, 1)` / `lead(x, 1)` use SQL's `LAG()`/`LEAD()` window functions instead — more efficient, but they blindly walk rows: if the result has a gap (e.g., a missing month), the previous row is the wrong period. Both approaches return NULL at the start of the window when there's no prior period in the data to match.

## The last() Transform

`last(metric)` returns the most recent time period's value, broadcast to every row. This **collapses the time dimension** — useful for comparing each period to the latest one, or for filtering.

For example, revenue by store alongside each store's most recent month's revenue:

In [5]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "ordered_at", "granularity": "quarter"}],
        "metrics": [
            "sum(order_total)",
            {"metric": "last(sum(order_total))", "alias": "latest_quarter"},
        ],
        "dimensions": ["stores.name"],
        "order": [{"field": "ordered_at", "direction": "desc"}],
        "limit": 10,
    }
)

print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'Latest Qtr':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    latest = row["orders.latest_quarter"]
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} ${latest:>13,.2f}")

Quarter      Store                       Revenue     Latest Qtr
--------------------------------------------------------------
2026-10      Philadelphia         $    89,681.41 $    89,681.41
2026-10      Chicago              $   128,273.95 $   128,273.95
2026-10      Brooklyn             $   148,714.36 $   148,714.36
2026-10      New Orleans          $    58,822.22 $    58,822.22
2026-10      San Francisco        $   113,035.91 $   113,035.91
2026-07      New Orleans          $    86,728.56 $    58,822.22
2026-07      Chicago              $   283,126.51 $   128,273.95
2026-07      Philadelphia         $   205,140.90 $    89,681.41
2026-07      San Francisco        $   248,237.65 $   113,035.91
2026-07      Brooklyn             $   333,996.21 $   148,714.36


## Composing Transforms in Filters

Transforms can be composed and used in filters. For example, `last(change(order_total_sum)) < 0` keeps only rows where the most recent period's change was negative — i.e., revenue declined in the latest period.

This is a powerful pattern: "show me all stores, but only those whose most recent quarter-over-quarter revenue went down":

In [6]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "ordered_at", "granularity": "quarter"}],
        "metrics": [
            "sum(order_total)",
            {"metric": "change(sum(order_total))", "alias": "qoq_change"},
        ],
        "dimensions": ["stores.name"],
        "filter": ["last(change(sum(order_total))) < 0"],
        "order": [{"field": "ordered_at", "direction": "desc"}],
        "limit": 10,
    }
)

print("Stores where latest QoQ revenue declined:")
print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'QoQ Change':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    chg = row["orders.qoq_change"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} {chg_str:>14}")

Stores where latest QoQ revenue declined:
Quarter      Store                       Revenue     QoQ Change
--------------------------------------------------------------
2026-10      San Francisco        $   113,035.91      $-135,202
2026-10      Philadelphia         $    89,681.41      $-115,459
2026-10      New Orleans          $    58,822.22       $-27,906
2026-10      Chicago              $   128,273.95      $-154,853
2026-10      Brooklyn             $   148,714.36      $-185,282
2026-07      Chicago              $   283,126.51        $-4,658
2026-07      Brooklyn             $   333,996.21       $-34,010
2026-07      Philadelphia         $   205,140.90       $-22,482
2026-07      New Orleans          $    86,728.56       $+61,789
2026-07      San Francisco        $   248,237.65       $-10,935


## Summary

Each of these concepts is simple and natural at the semantic level, but requires non-trivial SQL (CTEs, self-joins, window functions) to implement. SLayer offloads that effort.

| Transform | What it does | SQL implementation |
|-----------|-------------|-------------------|
| `change(x)` | Difference from previous period | Self-join CTE (gap-safe; first period is NULL) |
| `change_pct(x)` | Percentage change from previous period | Same as change, divided |
| `time_shift(x, -1, 'year')` | Same metric, shifted in time | Calendar-based self-join CTE |
| `lag(x)` / `lead(x)` | Previous/next row's value | SQL LAG/LEAD window (gap-blind) |
| `last(x)` | Most recent period's value, broadcast | Sub-query with FIRST_VALUE |
| `cumsum(x)` | Running total over time | SUM window function |

All transforms compose: `cumsum(change(x))`, `last(change(x))` in filters, etc.

See the [Expressions docs](../../concepts/expressions.md) for the full reference.